# Normalización completa del dataset Built

## 1. Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from difflib import get_close_matches

## 2. Cargar el dataset built a normalizar

In [2]:
# Definir la ruta del dataset Built normalizado
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\07 - Built\01 - built municipios normalizados.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

Filas cargadas: 2922240
Columnas disponibles: ['fecha', 'codigo_municipio', 'municipio', 'fraccion_construida', 'area_construida_km2', 'latitud', 'crecimiento_construido']
Tamaño del dataset: 2922240 filas x 7 columnas


,fecha,codigo_municipio,municipio,fraccion_construida,area_construida_km2,latitud,crecimiento_construido
0,2000-01-01,15001,abegondo,111.620694,9169.640000,5.144827e+06,NaN
1,2000-01-02,15001,abegondo,111.624217,9169.929447,5.144827e+06,NaN
2,2000-01-03,15001,abegondo,111.627741,9170.218894,5.144827e+06,NaN
3,2000-01-04,15001,abegondo,111.631264,9170.508342,5.144827e+06,NaN
4,2000-01-05,15001,abegondo,111.634787,9170.797789,5.144827e+06,NaN


In [3]:
# Verificar la estructura del dataset Built
print("=" * 60)
print("VERIFICACIÓN DE ESTRUCTURA DEL DATASET BUILT")
print("=" * 60)

# Verificar columnas esperadas
columnas_esperadas = ['fecha', 'codigo_municipio', 'municipio', 'fraccion_construida', 'area_construida_km2', 'crecimiento_construido', 'latitud']
columnas_presentes = df.columns.tolist()

print(f"✅ Columnas esperadas: {len(columnas_esperadas)}")
print(f"📋 Columnas presentes: {len(columnas_presentes)}")
print()

# Mostrar mapeo de columnas
for i, col in enumerate(columnas_presentes, 1):
    estado = "ok" if col in columnas_esperadas else "error"
    print(f"   {i}. {estado} {col}")

# Verificar tipos de datos
print(f"\nTIPOS DE DATOS:")
print("-" * 30)
for col in columnas_presentes:
    tipo = df[col].dtype
    nulos = df[col].isnull().sum()
    print(f"   • {col}: {tipo} ({nulos:,} nulos)")

# Verificar información temporal
if 'fecha' in df.columns:
    print(f"\nINFORMACIÓN TEMPORAL:")
    print(f"   • Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")
    print(f"   • Fechas únicas: {df['fecha'].nunique():,}")
else:
    print(f"\nColumna 'fecha' no encontrada")

print("=" * 60)

VERIFICACIÓN DE ESTRUCTURA DEL DATASET BUILT
✅ Columnas esperadas: 7
📋 Columnas presentes: 7

   1. ok fecha
   2. ok codigo_municipio
   3. ok municipio
   4. ok fraccion_construida
   5. ok area_construida_km2
   6. ok latitud
   7. ok crecimiento_construido

TIPOS DE DATOS:
------------------------------
   • fecha: object (0 nulos)
   • codigo_municipio: int64 (0 nulos)
   • municipio: object (0 nulos)
   • fraccion_construida: float64 (0 nulos)
   • area_construida_km2: float64 (0 nulos)
   • latitud: float64 (0 nulos)
   • crecimiento_construido: float64 (115,705 nulos)

INFORMACIÓN TEMPORAL:
   • Rango de fechas: 2000-01-01 a 2024-12-31
   • Fechas únicas: 9,132


In [4]:
# Normalizar la columna de fecha a tipo datetime y dejar solo la fecha (sin hora)
print("NORMALIZANDO COLUMNA DE FECHA")
print("-" * 40)

# Buscar la columna de fecha
col_fecha = None
if 'fecha' in df.columns:
    col_fecha = 'fecha'
elif 'date' in df.columns:
    col_fecha = 'date'
else:
    # Buscar columnas que contengan 'fecha' o 'date'
    for col in df.columns:
        if 'fecha' in col.lower() or 'date' in col.lower():
            col_fecha = col
            break
    
    # Si no se encuentra, usar la primera columna
    if col_fecha is None:
        col_fecha = df.columns[0]
        print(f"No se encontró columna de fecha, usando '{col_fecha}'")

print(f"Usando columna: '{col_fecha}'")
print(f"Tipo actual: {df[col_fecha].dtype}")
print(f"Ejemplos antes: {df[col_fecha].head(3).tolist()}")

# Convertir a datetime
df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

# Verificar valores nulos después de conversión
nulos_fecha = df[col_fecha].isnull().sum()
if nulos_fecha > 0:
    print(f"{nulos_fecha:,} fechas no pudieron convertirse (valores nulos)")

# Convertir a solo fecha (sin hora)
df[col_fecha] = df[col_fecha].dt.date

print(f"Conversión completada")
print(f"Ejemplos después: {df[col_fecha].dropna().astype(str).head(5).tolist()}")
print(f"Rango: {df[col_fecha].min()} a {df[col_fecha].max()}")
print(f"Fechas únicas: {df[col_fecha].nunique():,}")

NORMALIZANDO COLUMNA DE FECHA
----------------------------------------
Usando columna: 'fecha'
Tipo actual: object
Ejemplos antes: ['2000-01-01', '2000-01-02', '2000-01-03']
Conversión completada
Ejemplos después: ['2000-01-01', '2000-01-02', '2000-01-03', '2000-01-04', '2000-01-05']
Rango: 2000-01-01 a 2024-12-31
Fechas únicas: 9,132


In [5]:
# Filtrar registros solo entre el 1 de enero de 2000 y el 31 de diciembre de 2022
print("FILTRANDO PERÍODO TEMPORAL")
print("-" * 40)

fecha_inicio = pd.to_datetime('2000-01-01').date()
fecha_fin = pd.to_datetime('2022-12-31').date()

print(f"Registros antes del filtro: {len(df):,}")
print(f"Período a mantener: {fecha_inicio} a {fecha_fin}")

# Aplicar filtro
df_filtrado = df[(df[col_fecha] >= fecha_inicio) & (df[col_fecha] <= fecha_fin)]

# Mostrar estadísticas del filtrado
registros_eliminados = len(df) - len(df_filtrado)
print(f"Registros después del filtro: {len(df_filtrado):,}")
print(f"Registros eliminados: {registros_eliminados:,}")

if registros_eliminados > 0:
    print(f"Porcentaje conservado: {(len(df_filtrado)/len(df)*100):.2f}%")
    
    # Mostrar qué fechas se eliminaron
    fechas_eliminadas = df[~df.index.isin(df_filtrado.index)][col_fecha].dropna()
    if len(fechas_eliminadas) > 0:
        fecha_min_eliminada = fechas_eliminadas.min()
        fecha_max_eliminada = fechas_eliminadas.max()
        print(f"Fechas eliminadas van de: {fecha_min_eliminada} a {fecha_max_eliminada}")

# Actualizar el dataframe
df = df_filtrado

print(f"Filtrado completado")
print(f"Rango final: {df[col_fecha].min()} a {df[col_fecha].max()}")
print(f"Registros finales: {len(df):,}")

FILTRANDO PERÍODO TEMPORAL
----------------------------------------
Registros antes del filtro: 2,922,240
Período a mantener: 2000-01-01 a 2022-12-31
Registros después del filtro: 2,688,320
Registros eliminados: 233,920
Porcentaje conservado: 92.00%
Fechas eliminadas van de: 2023-01-01 a 2024-12-31
Filtrado completado
Rango final: 2000-01-01 a 2022-12-31
Registros finales: 2,688,320


In [6]:
# Guardar el dataset limpio en la ruta indicada
import os
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\07 - Built'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - built normalizado completo.csv')
df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset limpio guardado en: {archivo_export}')
print(f'Registros finales: {len(df):,}')
print(f'Período: {df[col_fecha].min()} a {df[col_fecha].max()}')

Dataset limpio guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\07 - Built\01 - built normalizado completo.csv
Registros finales: 2,688,320
Período: 2000-01-01 a 2022-12-31


In [7]:
# Resumen final del dataset Built normalizado
print("=" * 25)
print("   RESUMEN FINAL - DATASET BUILT NORMALIZADO")
print("=" * 25)

print(f"""
INFORMACIÓN FINAL:
   • Dataset: Built (superficies construidas de Galicia)
   • Registros totales: {len(df):,}
   • Período: {df[col_fecha].min()} a {df[col_fecha].max()}
   • Municipios únicos: {df['municipio'].nunique()}
   • Fechas únicas: {df[col_fecha].nunique():,}

COLUMNAS FINALES:
""")

for i, col in enumerate(df.columns, 1):
    tipo = df[col].dtype
    nulos = df[col].isnull().sum()
    unicos = df[col].nunique()
    print(f"   {i}. {col} ({tipo}) - {unicos:,} únicos, {nulos:,} nulos")

# Estadísticas de las variables numéricas principales
print(f"\nESTADÍSTICAS DE VARIABLES CLAVE:")
print("-" * 45)

if 'fraccion_construida' in df.columns:
    col_frac = 'fraccion_construida'
    valores_validos = df[col_frac].dropna()
    print(f"   • Fracción construida:")
    print(f"     - Min: {valores_validos.min():.4f}")
    print(f"     - Max: {valores_validos.max():.4f}")
    print(f"     - Media: {valores_validos.mean():.4f}")
    print(f"     - Valores > 0: {(valores_validos > 0).sum():,}")

if 'area_construida_km2' in df.columns:
    col_area = 'area_construida_km2'
    valores_area = df[col_area].dropna()
    print(f"   • Área construida (km²):")
    print(f"     - Min: {valores_area.min():.2f}")
    print(f"     - Max: {valores_area.max():.2f}")
    print(f"     - Media: {valores_area.mean():.2f}")
    print(f"     - Valores > 0: {(valores_area > 0).sum():,}")

print(f"""
CALIDAD DE DATOS:
   • Completitud temporal: 100% ({df[col_fecha].nunique():,} fechas de 2000-2022)
   • Cobertura municipal: {df['municipio'].nunique()}/92 municipios de Galicia
   • Densidad promedio: {len(df) / df[col_fecha].nunique():.1f} registros por fecha

ARCHIVO EXPORTADO:
   • Ubicación: {archivo_export}
   • Formato: CSV UTF-8
   • Tamaño: {len(df):,} registros × {len(df.columns)} columnas

LISTO PARA:
   • Combinación con otros datasets normalizados
   • Análisis exploratorio de datos (EDA)
   • Modelado predictivo de incendios
""")

print("=" * 25)
print("   NORMALIZACIÓN COMPLETA FINALIZADA")
print("=" * 25)

   RESUMEN FINAL - DATASET BUILT NORMALIZADO

INFORMACIÓN FINAL:
   • Dataset: Built (superficies construidas de Galicia)
   • Registros totales: 2,688,320
   • Período: 2000-01-01 a 2022-12-31
   • Municipios únicos: 315
   • Fechas únicas: 8,401

COLUMNAS FINALES:

   1. fecha (object) - 8,401 únicos, 0 nulos
   2. codigo_municipio (int64) - 317 únicos, 0 nulos
   3. municipio (object) - 315 únicos, 0 nulos
   4. fraccion_construida (float64) - 2,678,824 únicos, 0 nulos
   5. area_construida_km2 (float64) - 2,678,409 únicos, 0 nulos
   6. latitud (float64) - 319 únicos, 0 nulos
   7. crecimiento_construido (float64) - 2,571,885 únicos, 115,705 nulos

ESTADÍSTICAS DE VARIABLES CLAVE:
---------------------------------------------
   • Fracción construida:
     - Min: 0.0541
     - Max: 1897.3389
     - Media: 130.4034
     - Valores > 0: 2,688,320
   • Área construida (km²):
     - Min: 3.95
     - Max: 135940.86
     - Media: 8435.90
     - Valores > 0: 2,688,320

CALIDAD DE DATOS:
  